### ECSE415 - Introduction to Computer Vision
# Assignment 3
Anna Joy Aylward Burgess | 261124598 <br>
*March 18th, 2026*

## 0  Preparation



In [1]:
COLAB = False
# PS C:\Users\annaj\Documents\ECSE-415\ecse-415> .\ecse415\Scripts\Activate.ps1
#(ecse415) PS C:\Users\annaj\Documents\ECSE-415\ecse-415> 

if COLAB:
# mount drive
    from google.colab import drive
    drive.mount('/content/drive')

    # path variable to folder with images
    path = '/content/drive/My Drive/Colab Notebooks/ECSE 415 Assignments/As3/Q1/'
else:
    path = './Q1/'

# import libraries
import time
import numpy as np
import cv2
import matplotlib # Added import for the base matplotlib module
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import gc
import os
import pandas as pd
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import sys

print("External Libraries Used:")
print("NumPy version:", np.__version__)
print("OpenCV version:", cv2.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("Pandas version:", pd.__version__)
print("PyTorch version:", torch.__version__)

if not COLAB:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print("PyTorch version:", torch.__version__)
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Current Device: {torch.cuda.get_device_name(0)}")
        print(f"CUDA Capability: {torch.get_arch_list()}") # Look for 'sm_61' for GTX
    

External Libraries Used:
NumPy version: 2.4.3
OpenCV version: 4.13.0
Matplotlib version: 3.10.8
Pandas version: 3.0.1
PyTorch version: 2.11.0+cu126
Using device: cuda
PyTorch version: 2.11.0+cu126
CUDA Available: True
Current Device: NVIDIA GeForce GTX 1050


AttributeError: module 'torch' has no attribute 'get_arch_list'

---

## 1   Image Classification with Convolution Neural Network (CNN) and Naïve Bayes [40 points]

**In this part, you will classify MNIST digits into 10 categories using a CNN. You may choose to run the code on GPU.**




###1.1  Download Dataset [2 points]

**Use PyTorch class torchvision.datasets.MNIST to (down)load the dataset. Use batch size of 32.**




In [46]:
# 1. Define transformations (Convert to Tensor and Normalize)
# Mean and Std for MNIST are generally 0.1307 and 0.3081
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 2. Download/Load Training and Test Sets
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# 3. Create DataLoaders with batch size of 32
batch_size = 32

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nSuccessfully loaded {len(train_dataset)} training images and {len(test_dataset)} test images.")


Successfully loaded 60000 training images and 10000 test images.


###1.2  Implement a CNN with the layers mentioned below [5 points]

- **Convolution layer with 32 kernels of size 3×3**  
- **ReLU activation**  
- **Convolution layer with 64 kernels of size 3×3**  
- **ReLU activation**  
- **MaxPool layer with kernel size 2×2**  
- **Convolution layer with 64 kernels of size 3×3**  
- **ReLU activation**  
- **Convolution layer with 64 kernels of size 3×3**  
- **ReLU activation**  
- **Flatten layer (reshapes feature map into a vector of length 4096)**  
- **Linear layer with output size 10**




In [53]:
class MNIST_CNN(nn.Module):
    def __init__(self):
        super(MNIST_CNN, self).__init__()
        # 1. Conv layer: 32 kernels, 3x3 (Output: 32, 26, 26)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3)

        # 3. Conv layer: 64 kernels, 3x3 (Output: 64, 24, 24)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)

        # 5. Maxpool layer: 2x2 (Output: 64, 12, 12)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # 6. Conv layer: 64 kernels, 3x3 (Output: 64, 10, 10)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3)

        # 8. Conv layer: 64 kernels, 3x3 (Output: 64, 8, 8)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3)

        # 11. Linear layer: Output size 10
        # Flattening 64 * 8 * 8 = 4096
        self.fc = nn.Linear(4096, 10)

    def forward(self, x):
        # Layer 1 & 2: Conv + ReLU
        x = F.relu(self.conv1(x))
        # Layer 3 & 4: Conv + ReLU
        x = F.relu(self.conv2(x))
        # Layer 5: Maxpool
        x = self.pool(x)
        # Layer 6 & 7: Conv + ReLU
        x = F.relu(self.conv3(x))
        # Layer 8 & 9: Conv + ReLU
        x = F.relu(self.conv4(x))

        # Layer 10: Flattening
        x = x.view(-1, 4096)

        # Layer 11: Linear
        x = self.fc(x)
        return x

# Instantiate the model and move it to the GPU (T4 on Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MNIST_CNN().to(device)

print(model)

MNIST_CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (conv4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (fc): Linear(in_features=4096, out_features=10, bias=True)
)


###1.3  Create an instance of SGD optimizer with learning rate of 0.001. Use default settings for the remaining hyperparameters. Create an instance of categorical cross-entropy loss. [2 points]




In [48]:
# 1.3 Create an instance of categorical cross-entropy loss
criterion = nn.CrossEntropyLoss()

# Create an instance of SGD optimizer with learning rate of 0.001
# model.parameters() passes the weights of MNIST_CNN to the optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

print("Loss function: Categorical Cross-Entropy")
print(f"Optimizer: SGD (Learning Rate: 0.001)")

Loss function: Categorical Cross-Entropy
Optimizer: SGD (Learning Rate: 0.001)


###1.4  Train the CNN for 10 epochs. Display loss and accuracy at each training step. [4 points]




In [49]:
epochs = 10
model.train()

print(f"Starting training on {device}...\n")

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Update running loss and accuracy metrics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # --- Manual Progress Bar Logic ---
        # Calculate progress percentage
        progress = (i + 1) / len(train_loader)
        bar_length = 20
        filled_length = int(bar_length * progress)
        # Create string like [##########----------]
        bar = '█' * filled_length + '-' * (bar_length - filled_length)
        
        # Calculate current batch accuracy
        batch_acc = 100 * (predicted == labels).sum().item() / labels.size(0)
        
        # Print the bar. \r moves the cursor back to the start of the line.
        # end='' prevents a new line from being created.
        sys.stdout.write(f'\rEpoch [{epoch+1}/{epochs}] |{bar}| {100*progress:>3.0f}% '
                         f'Loss: {loss.item():.4f} Acc: {batch_acc:.1f}%')
        sys.stdout.flush()

    # Summary for the epoch
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    # Print a new line at the end of the epoch so the next epoch starts fresh
    print(f"\n=> Epoch {epoch+1} Complete: Avg Loss: {epoch_loss:.4f}, Total Acc: {epoch_acc:.2f}%\n")

print("Training Finished!")

Starting training on cpu...



Epoch [1/10] |██------------------|  13% Loss: 2.2882 Acc: 21.9%

KeyboardInterrupt: 